### Dataset

In [1]:
import dspy

train_data = [
    # RAG examples
    dspy.Example(
        question="What is the return window for unopened Beverages according to the product policy?",
        classification="rag",
    ).with_inputs("question"),
    dspy.Example(
        question="How many days can I return perishables like Produce or Seafood?",
        classification="rag",
    ).with_inputs("question"),
    dspy.Example(
        question="What categories are focused on during the Winter Classics 1997 campaign?",
        classification="rag",
    ).with_inputs("question"),
    dspy.Example(
        question="What is the definition of Average Order Value (AOV) in the KPI docs?",
        classification="rag",
    ).with_inputs("question"),
    dspy.Example(
        question="List all product categories from the catalog snapshot.",
        classification="rag",
    ).with_inputs("question"),
    dspy.Example(
        question="What is the return policy for non-perishables?",
        classification="rag",
    ).with_inputs("question"),
    dspy.Example(
        question="What are the dates for the Summer Beverages 1997 promotion?",
        classification="rag",
    ).with_inputs("question"),
    dspy.Example(
        question="How is Gross Margin calculated if cost is missing?",
        classification="rag",
    ).with_inputs("question"),
    dspy.Example(
        question="What notes are there for the Winter Classics 1997 in the marketing calendar?",
        classification="rag",
    ).with_inputs("question"),
    dspy.Example(
        question="Can opened Beverages be returned according to the policy?",
        classification="rag",
    ).with_inputs("question"),
    # Hybrid examples
    dspy.Example(
        question="What was the total quantity sold in the Beverages category during Summer Beverages 1997?",
        classification="hybrid",
    ).with_inputs("question"),
    dspy.Example(
        question="Using the AOV definition, what was the Average Order Value in 1997 for Dairy Products?",
        classification="hybrid",
    ).with_inputs("question"),
    dspy.Example(
        question="Who was the top customer by gross margin during Winter Classics 1997, approximating costs at 70% of UnitPrice?",
        classification="hybrid",
    ).with_inputs("question"),
    dspy.Example(
        question="What was the revenue from Confections during the dates of Winter Classics 1997?",
        classification="hybrid",
    ).with_inputs("question"),
    dspy.Example(
        question="Calculate the AOV for orders in the Summer Beverages 1997 period.",
        classification="hybrid",
    ).with_inputs("question"),
    dspy.Example(
        question="Top category by gross margin in 1997, using the KPI approximation for costs.",
        classification="hybrid",
    ).with_inputs("question"),
    dspy.Example(
        question="Total discounts given on Beverages and Condiments during Summer Beverages 1997.",
        classification="hybrid",
    ).with_inputs("question"),
    dspy.Example(
        question="What was the gross margin for Seafood category in the Winter Classics 1997 timeframe?",
        classification="hybrid",
    ).with_inputs("question"),
    dspy.Example(
        question="Using KPI defs, find AOV for customers from USA during Summer Beverages 1997.",
        classification="hybrid",
    ).with_inputs("question"),
    dspy.Example(
        question="Revenue from Dairy Products during the holiday gifting period in 1997.",
        classification="hybrid",
    ).with_inputs("question"),
    # SQL examples
    dspy.Example(
        question="What are the top 5 products by total revenue all-time?",
        classification="sql",
    ).with_inputs("question"),
    dspy.Example(
        question="How many orders were placed in 1997?",
        classification="sql",
    ).with_inputs("question"),
    dspy.Example(
        question="Who is the customer with the most orders?",
        classification="sql",
    ).with_inputs("question"),
    dspy.Example(
        question="What is the total quantity sold for ProductID 1?",
        classification="sql",
    ).with_inputs("question"),
    dspy.Example(
        question="List the top 3 countries by number of customers.",
        classification="sql",
    ).with_inputs("question"),
    dspy.Example(
        question="What is the average unit price across all products?",
        classification="sql",
    ).with_inputs("question"),
    dspy.Example(
        question="Total revenue from orders by EmployeeID 5.",
        classification="sql",
    ).with_inputs("question"),
    dspy.Example(
        question="How many distinct products were sold in orders from Germany?",
        classification="sql",
    ).with_inputs("question"),
    dspy.Example(
        question="What is the maximum discount given on any order detail?",
        classification="sql",
    ).with_inputs("question"),
    dspy.Example(
        question="Top supplier by number of products supplied.",
        classification="sql",
    ).with_inputs("question"),
]

### Model

In [2]:
import dspy
from dspy.teleprompt import BootstrapFewShot

MODEL_NAME = "ollama_chat/phi3.5:3.8b-mini-instruct-q4_K_M"
API_BASE = "http://localhost:11434"

lm = dspy.LM(MODEL_NAME, api_base=API_BASE, max_tokens=2048)
dspy.settings.configure(lm=lm)


class RouterSignature(dspy.Signature):
    """Classify the user query to decide the tool.
    - 'sql': precise data questions, aggregations, counting, or looking up DB records.
    - 'rag': questions about policies, marketing definitions, return windows, or text docs.
    - 'hybrid': requires both data lookup AND policy definitions (e.g. "Revenue during Summer Sale 1997").
    """

    question = dspy.InputField()
    classification = dspy.OutputField(
        desc="EXACTLY one of: sql, rag, hybrid. No other text."
    )


class Router(dspy.Module):
    def __init__(self):
        super().__init__()
        self.prog = dspy.ChainOfThought(RouterSignature)

    def forward(self, question):
        return self.prog(question=question)

### Datasplit

In [ ]:
import random
from collections import defaultdict

random.seed(123)

grouped = defaultdict(list)
for ex in train_data:
    grouped[ex.classification].append(ex)

trainset = []
testset = []

for cls, examples in grouped.items():
    random.shuffle(examples)
    trainset.extend(examples[:8])
    testset.extend(examples[8:])

random.shuffle(trainset)

### Evaluation

In [ ]:
def classification_metric(example, pred, trace=None):
    return example.classification.lower().strip() == pred.classification.lower().strip()


def evaluate_router(module, dataset, label="Generic"):
    print(f"\n--- Evaluating {label} ---")
    correct = 0
    total = len(dataset)

    for ex in dataset:
        pred = module(question=ex.question)
        passed = classification_metric(ex, pred)
        if passed:
            correct += 1
        print(
            f"Q: {ex.question[:40]}... | True: {ex.classification} | Pred: {pred.classification} | {'✅' if passed else '❌'}"
        )

    score = (correct / total) * 100
    print(f"Score: {score:.2f}%")
    return score

### Training Loop

In [4]:
print(f"Loaded {len(train_data)} examples.")

uncompiled_router = Router()
print("\nRunning Baseline (Uncompiled)...")
score_before = evaluate_router(uncompiled_router, testset, label="Baseline")

print("\nRunning Optimization (BootstrapFewShot)...")
teleprompter = BootstrapFewShot(
    metric=classification_metric,
    max_bootstrapped_demos=5,
    max_labeled_demos=5,
    max_rounds=5
)
with dspy.context(cached=False):
    compiled_router = teleprompter.compile(student=uncompiled_router, trainset=trainset)

print("\nRunning Optimized Model...")
score_after = evaluate_router(compiled_router, testset, label="Optimized")

print("\n" + "=" * 30)
print("OPTIMIZATION RESULTS")
print("=" * 30)
print(f"Metric: Classification Accuracy")
print(f"Before: {score_before:.2f}%")
print(f"After:  {score_after:.2f}%")
print(f"Delta:  {score_after - score_before:+.2f}%")

compiled_router.save("optimized_router.json")
print("\nOptimized router saved to 'optimized_router.json'")

Loaded 30 examples.

Running Baseline (Uncompiled)...

--- Evaluating Baseline ---
Q: List all product categories from the cat... | True: rag | Pred: sql | ❌
Q: What is the return window for unopened B... | True: rag | Pred: hybrid | ❌
Q: Revenue from Dairy Products during the h... | True: hybrid | Pred: sql | ❌
Q: Using KPI defs, find AOV for customers f... | True: hybrid | Pred: hybrid | ✅
Q: Who is the customer with the most orders... | True: sql | Pred: sql | ✅
Q: What is the total quantity sold for Prod... | True: sql | Pred: sql | ✅
Score: 50.00%

Running Optimization (BootstrapFewShot)...


 25%|██▌       | 6/24 [01:45<05:16, 17.58s/it]


Bootstrapped 5 full traces after 6 examples for up to 5 rounds, amounting to 10 attempts.

Running Optimized Model...

--- Evaluating Optimized ---
Q: List all product categories from the cat... | True: rag | Pred: sql | ❌
Q: What is the return window for unopened B... | True: rag | Pred: rag | ✅
Q: Revenue from Dairy Products during the h... | True: hybrid | Pred: hybrid | ✅
Q: Using KPI defs, find AOV for customers f... | True: hybrid | Pred: hybrid | ✅
Q: Who is the customer with the most orders... | True: sql | Pred: sql | ✅
Q: What is the total quantity sold for Prod... | True: sql | Pred: sql | ✅
Score: 83.33%

OPTIMIZATION RESULTS
Metric: Classification Accuracy
Before: 50.00%
After:  83.33%
Delta:  +33.33%

Optimized router saved to 'optimized_router.json'
